In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBClassifier
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train = pd.read_csv('/content/drive/MyDrive/GenAI_Project/data/Train.csv')
validation = pd.read_csv('/content/drive/MyDrive/GenAI_Project/data/Validation.csv')

print(train.shape)
print(train["Label"].value_counts())
print(train.head)

(4465, 129)
Label
0    3000
1    1465
Name: count, dtype: int64
<bound method NDFrame.head of         F1    F2    F3    F4   F5    F6  F7    F8   F9  F10  ...  F120  F121  \
0       82    47    41     3    0     3   0     3    0    0  ...     0     0   
1        5     0     0     0   30     5   0     0    0    0  ...     0     0   
2     4581  5175  1957  7073  470  2669   0  3553  132    0  ...     0     0   
3        6    38     0    18    1     1   1     2    4    1  ...     0     0   
4       88    48    44     4    0     4   0     4    0    0  ...     0     0   
...    ...   ...   ...   ...  ...   ...  ..   ...  ...  ...  ...   ...   ...   
4460     4     2     2     1    1     1   0     1    0    0  ...     0     0   
4461     6   914    85    34  160     5   1     2   74    1  ...     0     0   
4462     8    33     4    12    0     4   0     4    0    0  ...     0     0   
4463     5     0     0     0    0     5   0     0    0    0  ...     0     0   
4464   160    15     0    

In [4]:
X_train = train.drop('Label', axis=1).values
Y_train = train['Label'].values

X_val = validation.drop('ID', axis=1).values

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [6]:
#Train/test split for local evaluation
X_tr, X_test, y_tr, y_test = train_test_split(
    X_train_scaled, Y_train,
    test_size=0.2,
    random_state=42,
    stratify=Y_train
)

#Train classifier
classifier = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
classifier.fit(X_tr, y_tr)

#Local RMSE evaluation
preds = classifier.predict_proba(X_test)[:, 1]
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"Local RMSE: {rmse:.4f}")

# Retrain on full data before generating submission
classifier.fit(X_train_scaled, Y_train)
val_preds = classifier.predict_proba(X_val_scaled)[:, 1]

submission = pd.DataFrame({
    'ID': range(1, len(val_preds) + 1),
    'Label': val_preds
})
submission.to_csv('baseline_submission.csv', index=False)

Local RMSE: 0.1404


In [7]:
# Архітектура Генератора
class TabularGenerator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(TabularGenerator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# Архітектура Дискримінатора
class TabularDiscriminator(nn.Module):
    def __init__(self, input_dim):
        super(TabularDiscriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [8]:
# Головна функція для тренування та генерації
def train_tabular_gan_and_generate(X_minority, num_samples_to_generate,
                                   noise_dim=50, epochs=200, batch_size=64, lr=0.0002):
    input_dim = X_minority.shape[1]

    # Підготовка даних для PyTorch
    tensor_data = torch.tensor(X_minority, dtype=torch.float32)
    dataset = TensorDataset(tensor_data)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    # Ініціалізація моделей
    generator = TabularGenerator(noise_dim, input_dim)
    discriminator = TabularDiscriminator(input_dim)

    # Функція втрат та оптимізатори
    criterion = nn.BCELoss()
    optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    print(f"Початок тренування GAN на {epochs} епох...")

    for epoch in range(epochs):
        for i, data in enumerate(dataloader):
            real_data = data[0]
            current_batch_size = real_data.size(0)

            # Мітки для справжніх та фейкових даних
            real_labels = torch.ones(current_batch_size, 1)
            fake_labels = torch.zeros(current_batch_size, 1)

            # ---------------------
            # Тренування Дискримінатора
            # ---------------------
            optimizer_D.zero_grad()

            # На справжніх даних
            output_real = discriminator(real_data)
            loss_D_real = criterion(output_real, real_labels)

            # На фейкових даних
            noise = torch.randn(current_batch_size, noise_dim)
            fake_data = generator(noise)
            output_fake = discriminator(fake_data.detach())
            loss_D_fake = criterion(output_fake, fake_labels)

            # Оновлення ваг дискримінатора
            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            optimizer_D.step()

            # ---------------------
            # Тренування Генератора
            # ---------------------
            optimizer_G.zero_grad()

            # Генератор хоче "обдурити" дискримінатор
            output_fake_G = discriminator(fake_data)
            loss_G = criterion(output_fake_G, real_labels)

            # Оновлення ваг генератора
            loss_G.backward()
            optimizer_G.step()

        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Loss D: {loss_D.item():.4f}, Loss G: {loss_G.item():.4f}")

    print("Тренування завершено! Генеруємо нові дані...")

    # Генерація фінальних даних
    generator.eval()
    with torch.no_grad():
        final_noise = torch.randn(num_samples_to_generate, noise_dim)
        generated_data = generator(final_noise).numpy()

    return generated_data

In [9]:
# Відділяємо лише приклади з malware
MALWARE_LABEL = 1
malware_mask = (Y_train == MALWARE_LABEL)
X_malware = X_train_scaled[malware_mask]

# Визначаємо, скільки прикладів нам не вистачає для балансу
num_normal = sum(Y_train == 0)
num_malware = sum(Y_train == 1)
samples_to_add = num_normal - num_malware # Генеруємо стільки, щоб зрівняти класи

print(f"Кількість Normal: {num_normal}, Malware: {num_malware}")
print(f"Буде згенеровано {samples_to_add} нових прикладів malware.")

# Запускаємо GAN
if samples_to_add > 0:
    generated_malware = train_tabular_gan_and_generate(
        X_minority=X_malware,
        num_samples_to_generate=samples_to_add,
        epochs=200
    )

    # Додаємо згенеровані дані до оригінального датасету
    generated_labels = np.full(samples_to_add, MALWARE_LABEL)

    X_train_augmented = np.vstack((X_train_scaled, generated_malware))
    Y_train_augmented = np.hstack((Y_train, generated_labels))
else:
    print("Класи вже збалансовані або міноритарний клас більший.")
    X_train_augmented = X_train_scaled
    Y_train_augmented = Y_train

print(f"Новий розмір тренувального датасету: {X_train_augmented.shape}")

Кількість Normal: 3000, Malware: 1465
Буде згенеровано 1535 нових прикладів malware.
Початок тренування GAN на 200 епох...
Epoch [1/200] | Loss D: 1.3799, Loss G: 0.6907
Epoch [50/200] | Loss D: 1.2099, Loss G: 0.8832
Epoch [100/200] | Loss D: 1.2364, Loss G: 0.9403
Epoch [150/200] | Loss D: 1.1257, Loss G: 1.0314
Epoch [200/200] | Loss D: 1.1376, Loss G: 1.1098
Тренування завершено! Генеруємо нові дані...
Новий розмір тренувального датасету: (6000, 128)


In [10]:
# 1. Спліт збагачених даних для локальної перевірки
X_tr_aug, X_test_aug, y_tr_aug, y_test_aug = train_test_split(
    X_train_augmented, Y_train_augmented,
    test_size=0.2,
    random_state=42,
    stratify=Y_train_augmented
)

# 2. Ініціалізація та тренування класифікатора (XGBoost)
print("Навчання XGBoost на augmented даних...")
classifier_aug = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
classifier_aug.fit(X_tr_aug, y_tr_aug)

# 3. Локальна оцінка (RMSE)
preds_aug = classifier_aug.predict_proba(X_test_aug)[:, 1]
rmse_aug = np.sqrt(mean_squared_error(y_test_aug, preds_aug))
print(f"Local RMSE (with GAN): {rmse_aug:.4f}")

Навчання XGBoost на augmented даних...
Local RMSE (with GAN): 0.1274


In [11]:
# 1. Перенавчання на ВСІХ збагачених даних
print("Перенавчання на повному augmented датасеті...")
classifier_aug.fit(X_train_augmented, Y_train_augmented)

# 2. Передбачення для валідаційного (тестового) набору
print("Генерація передбачень...")
final_predictions = classifier_aug.predict_proba(X_val_scaled)[:, 1]

# 3. Формування файлу submission.csv
submission_df = pd.DataFrame({
    'ID': validation['ID'],
    'Label': final_predictions
})

# Збереження у файл
submission_filename = 'submission_gan.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"Файл {submission_filename} успішно створено!")

Перенавчання на повному augmented датасеті...
Генерація передбачень...
Файл submission_gan.csv успішно створено!


In [13]:
print(f"Базовий RMSE: {rmse:.4f}")
print(f"RMSE після GAN: {rmse_aug:.4f}")
print(f"Покращення: {abs(rmse - rmse_aug):.4f}")

Базовий RMSE: 0.1404
RMSE після GAN: 0.1274
Покращення: 0.0130
